<a href="https://colab.research.google.com/github/njadux/TextSummarizer-FineTuned-T5/blob/main/Text_Summarization_Using_Pre_trained_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip install transformers datasets evaluate rouge_score -q

In [14]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
import evaluate

    max_input_length	The maximum length (in tokens) of the source document (the text you are summarizing) that the model will accept. Longer inputs will be truncated (cut off).
    
    max_target_length	The maximum length (in tokens) of the generated summary (the output) that the model is allowed to produce.

In [15]:
model_name = 't5-small'
max_input_length = 512
max_target_length = 128
batch_size = 8
num_train_epochs = 2
output_dir = 't5-small-finetuned-summarization'

In [30]:
from datasets import load_dataset

data_files = {
    "train": "/content/train.csv",
    "validation": "/content/validation.csv",
    "test": "/content/test.csv"
}

# Attempt to load the original CSV files with error handling
dataset = load_dataset("csv", data_files=data_files, on_bad_lines='warn')
print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'article', 'highlights'],
        num_rows: 26330
    })
    validation: Dataset({
        features: ['id', 'article', 'highlights'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['id', 'article', 'highlights'],
        num_rows: 11490
    })
})


In [31]:
# Handle missing values by replacing them with empty strings within the dataset
# This step is no longer necessary as the data is loaded from clean files.
# def clean_dataset(example):
#     example['article'] = str(example['article']) if example['article'] is not None else ''
#     example['highlights'] = str(example['highlights']) if example['highlights'] is not None else ''
#     return example

# dataset = dataset.map(clean_dataset)

print("---> Dataset loaded and ready")
print(dataset['train'][0])

---> Dataset loaded and ready
{'id': '0001d1afc246a7964130f43ae940af6bc6c57f01', 'article': "By . Associated Press . PUBLISHED: . 14:11 EST, 25 October 2013 . | . UPDATED: . 15:36 EST, 25 October 2013 . The bishop of the Fargo Catholic Diocese in North Dakota has exposed potentially hundreds of church members in Fargo, Grand Forks and Jamestown to the hepatitis A virus in late September and early October. The state Health Department has issued an advisory of exposure for anyone who attended five churches and took communion. Bishop John Folda (pictured) of the Fargo Catholic Diocese in North Dakota has exposed potentially hundreds of church members in Fargo, Grand Forks and Jamestown to the hepatitis A . State Immunization Program Manager Molly Howell says the risk is low, but officials feel it's important to alert people to the possible exposure. The diocese announced on Monday that Bishop John Folda is taking time off after being diagnosed with hepatitis A. The diocese says he contrac

In [38]:
def clean_csv_safe(path):
    # Try reading, skip broken lines, fill missing values, specify encoding
    try:
        df = pd.read_csv(path, on_bad_lines="skip", engine="python", encoding='latin-1')
        df.fillna("", inplace=True)

        # Trim to expected columns if more appear
        expected_cols = ["article", "highlights"]
        # Ensure all expected columns exist, fill with empty string if not
        for col in expected_cols:
            if col not in df.columns:
                df[col] = ""
        df = df[expected_cols]


        cleaned_path = path.replace(".csv", "_clean.csv")
        df.to_csv(cleaned_path, index=False)
        print(f"✅ Cleaned and saved: {cleaned_path} (rows={len(df)})")
        return cleaned_path
    except UnicodeDecodeError:
        print(f"UnicodeDecodeError reading {path}. Trying 'cp1252' encoding.")
        try:
            df = pd.read_csv(path, on_bad_lines="skip", engine="python", encoding='cp1252')
            df.fillna("", inplace=True)
            for col in expected_cols:
                if col not in df.columns:
                    df[col] = ""
            df = df[expected_cols]
            cleaned_path = path.replace(".csv", "_clean.csv")
            df.to_csv(cleaned_path, index=False)
            print(f"✅ Cleaned and saved: {cleaned_path} (rows={len(df)}) with cp1252 encoding")
            return cleaned_path
        except Exception as e:
            print(f"Error reading {path} with 'cp1252' encoding: {e}")
            return None
    except Exception as e:
        print(f"Error reading {path}: {e}")
        return None


train_path = clean_csv_safe("/content/train.csv")
val_path = clean_csv_safe("/content/validation.csv")
test_path = clean_csv_safe("/content/test.csv")

# Now load the cleaned CSVs into a datasets DatasetDict
from datasets import load_dataset

data_files = {}
if train_path:
    data_files["train"] = train_path
if val_path:
    data_files["validation"] = val_path
if test_path:
    data_files["test"] = test_path

if data_files:
    dataset = load_dataset("csv", data_files=data_files)
    print("\n--- DatasetDict loaded from cleaned files ---")
    print(dataset)
else:
    print("\n--- No cleaned files were successfully loaded ---")

✅ Cleaned and saved: /content/train_clean.csv (rows=57559)
✅ Cleaned and saved: /content/validation_clean.csv (rows=13368)
✅ Cleaned and saved: /content/test_clean.csv (rows=11490)


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]


--- DatasetDict loaded from cleaned files ---
DatasetDict({
    train: Dataset({
        features: ['article', 'highlights'],
        num_rows: 57559
    })
    validation: Dataset({
        features: ['article', 'highlights'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights'],
        num_rows: 11490
    })
})


In [39]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess_function(examples):
    inputs = [str(x) for x in examples["article"]]
    targets = [str(x) for x in examples["highlights"]]
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=max_target_length, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Map:   0%|          | 0/57559 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/13368 [00:00<?, ? examples/s]

Map:   0%|          | 0/11490 [00:00<?, ? examples/s]

In [40]:
print(tokenized_datasets)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 57559
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 11490
    })
})


In [41]:
# MODEL + DATA COLLATOR
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [44]:
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    eval_strategy='epoch',  # Corrected argument name
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=num_train_epochs,
    predict_with_generate=True,
    fp16=True,
    logging_steps=100,
    push_to_hub=False,
    report_to="none"  # disable wandb by default
)

In [45]:
rouge = evaluate.load('rouge')

      # replace -100 in the labels as tokenizer.pad_token_id  -> is a required step to make the loss function ignore padding tokens during training, ensuring the model only learns to predict the actual summary text.
      
      	During Seq2Seq training, the loss function ignores (does not calculate error for) tokens labeled as -100. This line ensures that the special padding tokens used to make all target sequences the same length are correctly marked with -100 so the model doesn't try to learn them as part of the output summary.




In [46]:
# compute_metrics for Trainer uses generated text -> rouge

def postprocess_text(preds, labels):
  preds = [pred.strip() for pred in preds]
  labels = [label.strip() for label in labels]
  return preds, labels

def compute_metrics(eval_pred):
  predictions, labels = eval_pred

  # Debugging: Print predictions to inspect values
  print("Predictions during evaluation:", predictions)

  # decode
  decoded_preds = tokenizer.batch_decode(
      predictions,
      skip_special_tokens = True,
      clean_up_tokenization_spaces = True
  )

  # replace -100 in the labels as tokenizer.pad_token_id
  labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
  decoded_labels = tokenizer.batch_decode(
      labels,
      skip_special_tokens = True,
      clean_up_tokenization_spaces = True
  )

  decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

  result = rouge.compute(
      predictions = decoded_preds,
      references = decoded_labels,
      use_stemmer = True
  )

  result = {k: round(v * 100, 4) for k, v in result.items()}
  return result

In [47]:
trainer = Seq2SeqTrainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_datasets['train'].select(range(min(200, len(tokenized_datasets['train'])))), # select a smaller number of examples for training
    eval_dataset = tokenized_datasets['test'].select(range(min(200, len(tokenized_datasets['test'])))), # Use 'test' for evaluation during training
    tokenizer = tokenizer,
    data_collator = data_collator,
    compute_metrics = compute_metrics
)

/tmp/ipython-input-1991129401.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [51]:
trainer.train()

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,No log,1.976504,23.843900,11.077800,19.462000,19.522200
2,No log,1.970550,23.816100,10.897200,19.426100,19.489500


Predictions during evaluation: [[    0     3     9 ...  3127  7070    30]
 [    0 20492    83 ...     3     5     3]
 [    0 19034    23 ...    44 26238  6944]
 ...
 [    0 13439  4919 ...    26     8  9938]
 [    0   389  9351 ...  5840  4143     3]
 [    0 12976    31 ...    16  1882  7218]]
Predictions during evaluation: [[    0     3     9 ...  3127  7070    30]
 [    0 20492    83 ...     3     5     3]
 [    0 19034    23 ...    44 26238  6944]
 ...
 [    0 13439  4919 ...    26     8  9938]
 [    0   389  9351 ...  5840  4143     3]
 [    0 12976    31 ...    16  1882  7218]]


TrainOutput(global_step=50, training_loss=2.114845886230469, metrics={'train_runtime': 37.1355, 'train_samples_per_second': 10.771, 'train_steps_per_second': 1.346, 'total_flos': 54136720588800.0, 'train_loss': 2.114845886230469, 'epoch': 2.0})

In [52]:
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

('t5-small-finetuned-summarization/tokenizer_config.json',
 't5-small-finetuned-summarization/special_tokens_map.json',
 't5-small-finetuned-summarization/spiece.model',
 't5-small-finetuned-summarization/added_tokens.json',
 't5-small-finetuned-summarization/tokenizer.json')

In [54]:
from transformers import pipeline

# Load your fine-tuned model from output_dir
summarizer = pipeline(
    "summarization",
    model=output_dir,
    tokenizer=output_dir,
    framework="pt"  # or "tf" if you used TensorFlow backend
)

# Test 1️⃣ – a sample from your dataset
sample_text = dataset["test"][0]["article"]
print("Original article:\n", sample_text[:700], "...\n")

summary = summarizer(sample_text, max_length=128, min_length=30, do_sample=False)[0]['summary_text']
print("Generated summary:\n", summary)

Device set to use cuda:0
Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Original article:
 Ever noticed how plane seats appear to be getting smaller and smaller? With increasing numbers of people taking to the skies, some experts are questioning if having such packed out planes is putting passengers at risk. They say that the shrinking space on aeroplanes is not only uncomfortable - it's putting our health and safety in danger. More than squabbling over the arm rest, shrinking space on planes putting our health and safety in danger? This week, a U.S consumer advisory group set up by the Department of Transportation said at a public hearing that while the government is happy to set standards for animals flying on planes, it doesn't stipulate a minimum amount of space for humans. 'I ...

Generated summary:
 a consumer advisory group set up by the Department of Transportation said that while the government is happy to set standards for animals flying on planes, it doesn't stipulate a minimum amount of space for humans . but these tests are conducted using pla

In [56]:
custom_text = """
Apple Inc. unveiled the new iPhone 17 Pro Max during its annual launch event in California.
The phone introduces a titanium frame, improved battery efficiency, and a faster A19 Bionic chip.
Apple also announced new AI-powered features, including real-time translation and adaptive photography
modes that automatically adjust lighting based on scene context. Analysts predict strong early demand
despite rising prices, as Apple continues to dominate the premium smartphone market.
"""

print("Custom input:\n", custom_text)
summary = summarizer(custom_text, max_length=100, min_length=25, do_sample=False)[0]['summary_text']
print("\n Model Summary:\n", summary)

Your max_length is set to 100, but your input_length is only 94. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=47)
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Custom input:
 
Apple Inc. unveiled the new iPhone 17 Pro Max during its annual launch event in California.
The phone introduces a titanium frame, improved battery efficiency, and a faster A19 Bionic chip.
Apple also announced new AI-powered features, including real-time translation and adaptive photography
modes that automatically adjust lighting based on scene context. Analysts predict strong early demand
despite rising prices, as Apple continues to dominate the premium smartphone market.


 Model Summary:
 iPhone 17 Pro Max introduces titanium frame, improved battery efficiency, and faster A19 Bionic chip . AI-powered features include real-time translation and adaptive photography modes .
